Extractors

In [ ]:
# Choose ONE input method.
INPUT_MODE = "upload"   # "upload" or "drive"

# Only used if INPUT_MODE == "drive": folder that contains ZH, EN, DE (and .cupt files).
# DRIVE_DATA_DIR = "/content/drive/MyDrive/MWE2027/PARSEME_1.3"

INPUT_ROOT = "/content/parseme_input"
OUTPUT_DIR = "/content/parseme_lvc_output"
SAMPLE_PER_LANGUAGE_CATEGORY = 12

In [ ]:
from pathlib import Path
import shutil
import tarfile
import zipfile
from google.colab import files

input_root = Path(INPUT_ROOT)
input_root.mkdir(parents=True, exist_ok=True)


def safe_dest(base: Path, inner_name: str) -> Path:
    # Reject archive paths escaping the target directory.
    target = (base / inner_name).resolve()
    target.relative_to(base.resolve())
    return target


def unpack_archive(src: Path, dest: Path):
    dest.mkdir(parents=True, exist_ok=True)
    lower = src.name.lower()
    if lower.endswith(".zip"):
        with zipfile.ZipFile(src) as zf:
            for member in zf.infolist():
                if member.is_dir() or member.filename.startswith("__MACOSX/"):
                    continue
                target = safe_dest(dest, member.filename)
                target.parent.mkdir(parents=True, exist_ok=True)
                with zf.open(member) as reader, target.open("wb") as writer:
                    shutil.copyfileobj(reader, writer)
    elif lower.endswith((".tar.gz", ".tgz", ".tar")):
        with tarfile.open(src, "r:*") as tf:
            for member in tf:
                if not member.isfile():  # Never extract symlinks.
                    continue
                target = safe_dest(dest, member.name)
                target.parent.mkdir(parents=True, exist_ok=True)
                reader = tf.extractfile(member)
                if reader is not None:
                    with reader, target.open("wb") as writer:
                        shutil.copyfileobj(reader, writer)
    elif lower.endswith(".cupt"):
        shutil.copy2(src, dest / src.name)
    else:
        raise ValueError(f"Unsupported input: {src.name}. Upload ZIP / TAR.GZ / TGZ / CUPT.")


if INPUT_MODE == "upload":
    uploads = files.upload()  # Select ZH/EN/DE archive files together.
    if not uploads:
        raise RuntimeError("No files uploaded.")
    for name, content in uploads.items():
        archive = input_root / Path(name).name
        archive.write_bytes(content)
        dest = input_root / f"extracted_{archive.name.replace('.', '_')}"
        unpack_archive(archive, dest)
        print(f"Extracted {archive.name} to {dest}")
    DATA_ROOT = input_root
elif INPUT_MODE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_ROOT = Path(DRIVE_DATA_DIR)
    if not DATA_ROOT.is_dir():
        raise FileNotFoundError(f"Drive directory does not exist: {DATA_ROOT}")
    # Drive option expects already-unzipped .cupt files under ZH/EN/DE directories.
else:
    raise ValueError("INPUT_MODE must be 'upload' or 'drive'.")

cupt_paths = sorted(Path(DATA_ROOT).rglob("*.cupt"))
print(f"Found {len(cupt_paths)} .cupt files before gold/language filtering.")
for p in cupt_paths[:25]:
    print("  ", p.relative_to(DATA_ROOT))
if not cupt_paths:
    raise RuntimeError("No .cupt files found. Check the archive contents or Drive directory.")


Saving DE.tgz to DE.tgz
Saving EN.tgz to EN.tgz
Saving ZH.tgz to ZH.tgz
Extracted DE.tgz to /content/parseme_input/extracted_DE_tgz
Extracted EN.tgz to /content/parseme_input/extracted_EN_tgz
Extracted ZH.tgz to /content/parseme_input/extracted_ZH_tgz
Found 9 .cupt files before gold/language filtering.
   extracted_DE_tgz/DE/dev.cupt
   extracted_DE_tgz/DE/test.cupt
   extracted_DE_tgz/DE/train.cupt
   extracted_EN_tgz/EN/dev.cupt
   extracted_EN_tgz/EN/test.cupt
   extracted_EN_tgz/EN/train.cupt
   extracted_ZH_tgz/ZH/dev.cupt
   extracted_ZH_tgz/ZH/test.cupt
   extracted_ZH_tgz/ZH/train.cupt


In [ ]:
%%writefile /content/parseme13_colab_extractor.py
#!/usr/bin/env python3
"""Extract annotated LVC.full and LVC.cause from PARSEME 1.3 ZH/EN/DE .cupt.

Runs in Python 3 standard library, including Google Colab.
Keeps official train/dev/test split, annotated token IDs and full UD records.
Does not infer event configurations, implicit v heads, or negative examples.
"""

import argparse
import csv
import json
import random
import re
import sys
from collections import Counter, defaultdict
from pathlib import Path

LANGS = ('ZH', 'EN', 'DE')
CATEGORIES = ('LVC.full', 'LVC.cause')
UD_NAMES = ('id', 'form', 'lemma', 'upos', 'xpos', 'feats', 'head',
            'deprel', 'deps', 'misc', 'parseme_mwe')
COLUMNS = (
    'language', 'split', 'source_file', 'sent_id', 'instance_id',
    'category', 'sentence', 'mwe_id', 'mwe_token_ids',
    'mwe_surface', 'mwe_lemmas', 'mwe_upos', 'mwe_deprels',
    'mwe_heads', 'n_members', 'is_contiguous', 'ud_members_complete',
    'candidate_complement_type', 'candidate_argument_relation',
    'candidate_alternative_analysis', 'review_note',
)


def detect_language(path):
    """Infer language from a directory or filename segment such as /ZH/ or corpus_en.zip/."""
    for part in reversed(path.parts):
        for item in re.split(r'[^A-Za-z]+', part.upper()):
            if item in LANGS:
                return item
    return None


def detect_split(path):
    for item in re.split(r'[^a-z]+', path.stem.lower()):
        if item in ('train', 'dev', 'test'):
            return item
    return None


def is_eligible(path):
    forbidden = ('blind', 'trial', 'system', 'predicted', 'prediction')
    # Exclude non-gold or demonstration .cupt files regardless of nesting.
    lower = '/'.join(path.parts).lower()
    return (path.suffix.lower() == '.cupt'
            and detect_language(path) is not None
            and detect_split(path) is not None
            and not any(w in lower for w in forbidden))


def read_sentences(path):
    comments, tokens, ordinal = [], [], 0
    with path.open('r', encoding='utf-8-sig') as fh:
        for line_no, raw in enumerate(fh, 1):
            line = raw.rstrip('\r\n')
            if not line.strip():
                if tokens:
                    ordinal += 1
                    yield comments, tokens, ordinal
                comments, tokens = [], []
                continue
            if line.startswith('#'):
                comments.append(line)
                continue
            fields = line.split('\t')
            if len(fields) != 11:
                raise ValueError(
                    f'{path}:{line_no}: expected 11 tab-separated columns, '
                    f'found {len(fields)}. Is this an annotated .cupt file?'
                )
            # CoNLL-U surface ranges (1-2) and empty nodes (2.1) are not real UD words.
            if fields[0].isdigit():
                tokens.append(dict(zip(UD_NAMES, fields)))
    if tokens:
        yield comments, tokens, ordinal + 1


def meta(comments, key):
    pat = re.compile(r'^#\s*' + re.escape(key) + r'\s*=\s*(.*)$')
    for c in comments:
        found = pat.match(c)
        if found:
            return found.group(1)
    return ''


def reconstruct(tokens, lang):
    # Prefer the corpus-provided '# text'; this is only a fallback.
    chunks = []
    for idx, token in enumerate(tokens):
        if idx and lang != 'ZH':
            prev_misc = tokens[idx - 1]['misc'].split('|')
            if 'SpaceAfter=No' not in prev_misc:
                chunks.append(' ')
        chunks.append(token['form'])
    return ''.join(chunks)


def mwe_groups(tokens, loc):
    groups = defaultdict(lambda: {'category': None, 'members': []})
    for token in tokens:
        cell = token['parseme_mwe']
        if cell in ('*', '_', ''):
            continue
        for part in cell.split(';'):
            part = part.strip()
            if part in ('*', '_', ''):
                continue
            m = re.fullmatch(r'(\d+)(?::([A-Za-z][A-Za-z0-9_.-]*))?', part)
            if m is None:
                raise ValueError(f'{loc}: invalid PARSEME:MWE value {cell!r}')
            group_id, new_category = m.groups()
            group = groups[group_id]
            if new_category is not None:
                if group['category'] is not None and group['category'] != new_category:
                    raise ValueError(f'{loc}: inconsistent category for MWE {group_id}')
                group['category'] = new_category
            group['members'].append(token)
    return groups


def write_csv(path, fieldnames, rows):
    # utf-8-sig makes the CSV readable in Excel without mojibake for Chinese.
    with path.open('w', encoding='utf-8-sig', newline='') as fh:
        writer = csv.DictWriter(fh, fieldnames=fieldnames, extrasaction='ignore')
        writer.writeheader()
        writer.writerows(rows)


def run(root, output, sample_per_category=12):
    root = Path(root).resolve()
    output = Path(output).resolve()
    if not root.is_dir():
        raise FileNotFoundError(f'Cannot find data directory: {root}')
    output.mkdir(parents=True, exist_ok=True)

    all_cupt = list(root.rglob('*.cupt'))
    files = sorted(f for f in all_cupt if is_eligible(f))
    if not files:
        raise RuntimeError(
            f'No labeled ZH/EN/DE train/dev/test .cupt files found in {root}. '
            'Check that each corpus is inside a ZH, EN or DE named folder.'
        )

    rows = []
    counts = Counter()
    lemma_counts = Counter()
    manifest = []
    jsonl_path = output / 'lvc_instances.jsonl'
    with jsonl_path.open('w', encoding='utf-8') as handle:
        for file in files:
            lang, split = detect_language(file), detect_split(file)
            source_file = file.relative_to(root).as_posix()
            n_sentences = 0
            n_lvc = 0
            for comments, tokens, sequence in read_sentences(file):
                n_sentences += 1
                sent_id = meta(comments, 'sent_id') or f'ordinal_{sequence:08d}'
                sentence = meta(comments, 'text') or reconstruct(tokens, lang)
                groups = mwe_groups(tokens, f'{source_file} / {sent_id}')
                for group_id, group in sorted(groups.items(), key=lambda x: int(x[0])):
                    category = group['category']
                    if category not in CATEGORIES:
                        continue
                    members = sorted(group['members'], key=lambda t: int(t['id']))
                    ids = [int(t['id']) for t in members]
                    lemmas = ' | '.join(
                        t['lemma'] if t['lemma'] != '_' else t['form']
                        for t in members
                    )
                    # Position-based ID avoids collisions when sent_id is reused.
                    instance_id = f'{lang}:{split}:{source_file}:s{sequence}:mwe{group_id}'
                    row = {
                        'language': lang,
                        'split': split,
                        'source_file': source_file,
                        'sent_id': sent_id,
                        'instance_id': instance_id,
                        'category': category,
                        'sentence': sentence,
                        'mwe_id': group_id,
                        'mwe_token_ids': ','.join(map(str, ids)),
                        'mwe_surface': ' | '.join(t['form'] for t in members),
                        'mwe_lemmas': lemmas,
                        'mwe_upos': ' | '.join(t['upos'] for t in members),
                        'mwe_deprels': ' | '.join(t['deprel'] for t in members),
                        'mwe_heads': ' | '.join(t['head'] for t in members),
                        'n_members': len(members),
                        'is_contiguous': max(ids) - min(ids) + 1 == len(ids),
                        'ud_members_complete': all(
                            t['head'] != '_' and t['deprel'] != '_' for t in members
                        ),
                        'candidate_complement_type': '',
                        'candidate_argument_relation': '',
                        'candidate_alternative_analysis': '',
                        'review_note': '',
                    }
                    rows.append(row)
                    counts[lang, split, category] += 1
                    lemma_counts[lang, category, lemmas] += 1
                    handle.write(json.dumps(
                        {**row, 'ud_tokens': tokens, 'sentence_comments': comments},
                        ensure_ascii=False
                    ) + '\n')
                    n_lvc += 1
            manifest.append({
                'language': lang, 'split': split, 'source_file': source_file,
                'sentences_read': n_sentences, 'lvc_extracted': n_lvc,
            })
            print(f'{source_file} -> {n_sentences} sentences; {n_lvc} LVC', flush=True)

    write_csv(output / 'lvc_instances.csv', COLUMNS, rows)
    write_csv(output / 'source_manifest.csv',
              ('language', 'split', 'source_file', 'sentences_read', 'lvc_extracted'),
              manifest)
    count_rows = [
        {'language': lang, 'split': split, 'category': category,
         'occurrences': counts[lang, split, category]}
        for lang in LANGS
        for split in ('train', 'dev', 'test')
        for category in CATEGORIES
    ]
    write_csv(output / 'lvc_counts.csv',
              ('language', 'split', 'category', 'occurrences'), count_rows)
    lemma_rows = [
        {'language': lang, 'category': category, 'mwe_lemmas': lemmas,
         'occurrences': n}
        for (lang, category, lemmas), n in sorted(
            lemma_counts.items(), key=lambda item: (-item[1], item[0])
        )
    ]
    write_csv(output / 'lvc_lemma_types.csv',
              ('language', 'category', 'mwe_lemmas', 'occurrences'), lemma_rows)
    rng = random.Random(2027)
    sample_rows = []
    for lang in LANGS:
        for category in CATEGORIES:
            group = [row for row in rows if row['language'] == lang
                     and row['category'] == category]
            sample_rows += rng.sample(group, min(sample_per_category, len(group)))
    rng.shuffle(sample_rows)
    write_csv(output / 'lvc_review_sample.csv', COLUMNS, sample_rows)

    missing = [lang for lang in LANGS
               if not any(m['language'] == lang for m in manifest)]
    print('\n=== Extraction summary ===')
    for lang in LANGS:
        full = sum(counts[lang, s, 'LVC.full'] for s in ('train', 'dev', 'test'))
        cause = sum(counts[lang, s, 'LVC.cause'] for s in ('train', 'dev', 'test'))
        print(f'  {lang}: LVC.full={full}, LVC.cause={cause}, total={full+cause}')
    print(f'  TOTAL: {len(rows)}')
    if missing:
        print(f'WARNING: These languages were not found: {missing}', file=sys.stderr)
    print(f'Files saved in {output}')
    return len(rows)


def main():
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument('root', type=Path)
    p.add_argument('--out', type=Path, default=Path('parseme_lvc_output'))
    p.add_argument('--sample-per-category', type=int, default=12)
    args = p.parse_args()
    run(args.root, args.out, sample_per_category=args.sample_per_category)


if __name__ == '__main__':
    main()

Writing /content/parseme13_colab_extractor.py


In [ ]:
import subprocess
import sys

cmd = [
    sys.executable, "/content/parseme13_colab_extractor.py", str(DATA_ROOT),
    "--out", OUTPUT_DIR,
    "--sample-per-category", str(SAMPLE_PER_LANGUAGE_CATEGORY),
]
subprocess.run(cmd, check=True)

CompletedProcess(args=['/usr/bin/python3', '/content/parseme13_colab_extractor.py', '/content/parseme_input', '--out', '/content/parseme_lvc_output', '--sample-per-category', '12'], returncode=0)

In [ ]:
import pandas as pd
from IPython.display import display
from pathlib import Path

out = Path(OUTPUT_DIR)
counts = pd.read_csv(out / "lvc_counts.csv", encoding="utf-8-sig")
instances = pd.read_csv(out / "lvc_instances.csv", encoding="utf-8-sig")
manifest = pd.read_csv(out / "source_manifest.csv", encoding="utf-8-sig")

print("=== By language and LVC type ===")
display(counts.pivot_table(index="language", columns="category",
                           values="occurrences", aggfunc="sum", fill_value=0))
print(f"Total LVC occurrences extracted: {len(instances):,}")
print("=== Source files ===")
display(manifest)
print("=== First examples ===")
display(instances[["language", "split", "category", "mwe_surface",
                   "sentence", "mwe_deprels", "is_contiguous"]].head(12))

# Reference totals from the PARSEME 1.3 paper, only as a check, not a hard validation.
expected = {"ZH": 1391, "EN": 384, "DE": 344}
observed = instances.groupby("language").size().to_dict()
for lang, target in expected.items():
    current = int(observed.get(lang, 0))
    print(f"{lang}: extracted {current}, published reference {target} "
          + ("(matched)" if current == target else "(please check version/files/splits)"))

=== By language and LVC type ===


category,LVC.cause,LVC.full
language,,
DE,33,311
EN,51,333
ZH,177,1214


Total LVC occurrences extracted: 2,119
=== Source files ===


,language,split,source_file,sentences_read,lvc_extracted
0,DE,dev,extracted_DE_tgz/DE/dev.cupt,628,28
1,DE,test,extracted_DE_tgz/DE/test.cupt,1893,71
2,DE,train,extracted_DE_tgz/DE/train.cupt,6475,245
3,EN,dev,extracted_EN_tgz/EN/dev.cupt,1302,73
4,EN,test,extracted_EN_tgz/EN/test.cupt,3984,201
5,EN,train,extracted_EN_tgz/EN/train.cupt,2150,110
6,ZH,dev,extracted_ZH_tgz/ZH/dev.cupt,1215,33
7,ZH,test,extracted_ZH_tgz/ZH/test.cupt,3611,99
8,ZH,train,extracted_ZH_tgz/ZH/train.cupt,44103,1259


=== First examples ===


,language,split,category,mwe_surface,sentence,mwe_deprels,is_contiguous
0,DE,dev,LVC.full,Spaß | hat,Spaß hat auch Adrian beim Fechten.,obj | root,True
1,DE,dev,LVC.full,laufen | Gespräche,Den Angaben zufolge laufen derzeit Gespräche m...,root | nsubj,False
2,DE,dev,LVC.full,im | Einsatz | waren,Die Siegerehrungen nahmen Kreisjugendwart Diet...,case | obl | advcl,True
3,DE,dev,LVC.full,Wirkung | entfalten,"Dieser unplakative, abstrakte Umgang mit der F...",obj | root,True
4,DE,dev,LVC.cause,Zugriffsrechte | erteilen,Verwalten Sie Ihre Bekanntschaften in Listen (...,obj | acl,True
5,DE,dev,LVC.full,Job-Angebot | gemacht,Er gilt als russischer Mark Zuckerberg: Der Gr...,obj | parataxis,True
6,DE,dev,LVC.cause,gewährte | Einblick,Als Rahmenprogramm gewährte die Kriminalpolize...,root | obj,False
7,DE,dev,LVC.full,gaben | Konzert,"Die ""Höhner""-Musiker Hannes Schöner und Jens S...",root | obj,False
8,DE,dev,LVC.cause,unter | Strafe | stellte,"``Deutschland war das erste europäische Land, ...",case | obl | acl,True
9,DE,dev,LVC.full,in | Zusammenhang | stehen,"Eine Besonderheit des Verlages ist es, dass er...",case | obl | acl,True


ZH: extracted 1391, published reference 1391 (matched)
EN: extracted 384, published reference 384 (matched)
DE: extracted 344, published reference 344 (matched)


In [ ]:
from google.colab import files
import shutil
from pathlib import Path

archive = shutil.make_archive("/content/parseme_lvc_output", "zip", root_dir=OUTPUT_DIR)
print("Output archive:", archive)
for p in sorted(Path(OUTPUT_DIR).iterdir()):
    print(f"  {p.name}: {p.stat().st_size/1024:.1f} KB")
files.download(archive)

Output archive: /content/parseme_lvc_output.zip
  lvc_counts.csv: 0.4 KB
  lvc_instances.csv: 704.9 KB
  lvc_instances.jsonl: 12016.6 KB
  lvc_lemma_types.csv: 46.2 KB
  lvc_review_sample.csv: 25.6 KB
  source_manifest.csv: 0.5 KB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Lexical baseline

In [3]:

# ====================================================
# MWE 2027 | A2 LEXICAL BASELINE | v2.2 FINAL
# Input: reviewed Excel + PARSEME output ZIP
# No Google Drive / No GPU
# ====================================================

from google.colab import files
from pathlib import Path
from collections import Counter, defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer
import csv, io, re, math, zipfile
import xml.etree.ElementTree as ET

print("请同时上传最终审核 Excel 和 PARSEME ZIP")
uploaded = files.upload()

EXCEL = next(
    Path(n) for n in uploaded
    if n.lower().endswith(".xlsx")
)
CORPUS = next(
    Path(n) for n in uploaded
    if n.lower().endswith(".zip")
)

OUT = Path("/content/a2_results")
OUT.mkdir(exist_ok=True)

# ----------------------------------------------------
# 1. 读取 Excel 第一张工作表
# ----------------------------------------------------

def read_excel(p):
    ns = "{http://schemas.openxmlformats.org/spreadsheetml/2006/main}"

    with zipfile.ZipFile(p) as z:
        shared = []

        if "xl/sharedStrings.xml" in z.namelist():
            root = ET.fromstring(
                z.read("xl/sharedStrings.xml")
            )
            shared = [
                "".join(t.text or "" for t in e.iter(ns+"t"))
                for e in root.findall(ns+"si")
            ]

        root = ET.fromstring(
            z.read("xl/worksheets/sheet1.xml")
        )

    rows = []

    for row in root.iter(ns+"row"):
        values = {}

        for c in row.findall(ns+"c"):
            letters = re.match(
                r"[A-Z]+", c.attrib["r"]
            ).group()

            idx = 0
            for ch in letters:
                idx = idx * 26 + ord(ch) - 64
            idx -= 1

            kind = c.attrib.get("t")
            v = c.find(ns+"v")

            if kind == "inlineStr":
                value = "".join(
                    t.text or "" for t in c.iter(ns+"t")
                )
            elif v is None:
                value = ""
            elif kind == "s":
                value = shared[int(v.text)]
            else:
                value = v.text or ""

            values[idx] = value

        rows.append(values)

    headers = rows[0]

    return [
        {
            h: row.get(i, "")
            for i, h in headers.items()
            if h
        }
        for row in rows[1:]
    ]


raw = read_excel(EXCEL)
stimuli = []

# 自动提取73条主实验记录
# 19条补充材料不进入A2

for r in raw:
    task = r.get("task_type")

    if task == "CONFIG_3X3":
        item = dict(
            item_id=r["review_id"],
            task_type="CONFIG_3x3",
            language=r["language"],
            group_id=r["triplet_id"],
            context_id=r["context_config"],
            candidate_id=r["candidate_config"],
            preferred=r["preferred"],
        )

    elif task == "CONTEXT_SWAP_2x2":
        item = dict(
            item_id=r["review_id"],
            task_type=task,
            language=r["language"],
            group_id=r["family_id"],
            context_id=r["context_id"],
            candidate_id=r["candidate_id"],
            preferred=r["intended_preferred"],
        )

    else:
        continue

    item.update(
        context=r["context"],
        completion=r["completion"],
        native_review=r["native_review"],
        item_validated=r["item_validated"],
    )

    assert item["item_validated"] == "YES"
    assert item["native_review"] in ("ACCEPT", "REVISE")
    assert item["preferred"] in ("YES", "NO")
    assert all(
        item[k] for k in
        ("context", "completion", "group_id",
         "context_id", "candidate_id")
    )

    stimuli.append(item)

assert len(stimuli) == 73

print("Excel读取成功")
print("总审核记录：", len(raw))
print("A2评分记录：", len(stimuli))
print("3×3：", sum(
    r["task_type"] == "CONFIG_3x3" for r in stimuli
))
print("2×2：", sum(
    r["task_type"] == "CONTEXT_SWAP_2x2" for r in stimuli
))

# ----------------------------------------------------
# 2. 读取 PARSEME 训练集
# ----------------------------------------------------

with zipfile.ZipFile(CORPUS) as z:
    with z.open("lvc_instances.csv") as f:
        corpus_rows = list(csv.DictReader(
            io.TextIOWrapper(f, encoding="utf-8-sig")
        ))

train = defaultdict(dict)

for r in corpus_rows:
    if r.get("split", "").lower() != "train":
        continue

    sentence = (r.get("sentence") or "").strip()

    if sentence:
        key = (
            r.get("source_file", ""),
            r.get("sent_id", ""),
            sentence,
        )
        train[r["language"]][key] = sentence

print("\nPARSEME train句子数：")
for lang in ("ZH", "EN", "DE"):
    print(lang, len(train[lang]))

# ----------------------------------------------------
# 3. 基线一：Character Bigram PRIOR
# ----------------------------------------------------

def norm(s):
    return re.sub(r"\s+", " ", s.casefold().strip())


class BigramPrior:

    def __init__(self, texts, alpha=0.3):
        self.a = alpha
        self.uni = Counter()
        self.bi = Counter()
        vocab = set()

        for text in texts:
            chars = list(norm(text))
            vocab.update(chars)
            previous = "\x02"

            for ch in chars + ["\x03"]:
                self.uni[previous] += 1
                self.bi[previous, ch] += 1
                previous = ch

        self.v = len(vocab) + 2

    def score(self, text):
        chars = list(norm(text)) + ["\x03"]
        previous = "\x02"
        total = 0.0

        for ch in chars:
            prob = (
                self.bi[previous, ch] + self.a
            ) / (
                self.uni[previous] + self.a * self.v
            )

            total += math.log(prob)
            previous = ch

        return total / len(chars)


# ----------------------------------------------------
# 4. 基线二：TF-IDF Lexical Overlap
# ----------------------------------------------------

for lang in ("ZH", "EN", "DE"):

    texts = list(train[lang].values())
    assert texts, f"No training data for {lang}"

    records = [
        r for r in stimuli
        if r["language"] == lang
    ]

    vectorizer = TfidfVectorizer(
        analyzer="char",
        ngram_range=(1, 2),
        sublinear_tf=True,
    )

    vectorizer.fit(texts)
    prior = BigramPrior(texts)

    contexts = vectorizer.transform(
        [r["context"] for r in records]
    )

    completions = vectorizer.transform(
        [r["completion"] for r in records]
    )

    similarities = contexts.multiply(
        completions
    ).sum(axis=1)

    for i, r in enumerate(records):
        r["score_prior"] = round(
            prior.score(r["completion"]), 9
        )
        r["score_lexical_overlap"] = round(
            float(similarities[i, 0]), 9
        )

print("\n两种词汇基线评分完成")

# ----------------------------------------------------
# 5. 检查3×3与2×2矩阵完整性
# ----------------------------------------------------

GRIDS = {
    "CONFIG_3x3": (
        ["STATE", "BECOME", "CAUSE"],
        ["STATE", "BECOME", "CAUSE"],
    ),
    "CONTEXT_SWAP_2x2": (
        ["CA", "CB"],
        ["A", "B"],
    )
}

groups = defaultdict(list)

for r in stimuli:
    key = (
        r["task_type"],
        r["language"],
        r["group_id"],
    )
    groups[key].append(r)

by_context = []
by_family = []

for (task, lang, gid), rows in sorted(groups.items()):

    contexts, candidates = GRIDS[task]

    expected = {
        (c, k)
        for c in contexts for k in candidates
    }

    observed = {
        (r["context_id"], r["candidate_id"])
        for r in rows
    }

    assert observed == expected
    assert len(rows) == len(expected)

    for c in contexts:
        sub = [
            r for r in rows if r["context_id"] == c
        ]
        assert len({r["context"] for r in sub}) == 1
        assert sum(
            r["preferred"] == "YES" for r in sub
        ) == 1

    for k in candidates:
        assert len({
            r["completion"] for r in rows
            if r["candidate_id"] == k
        }) == 1

    for method, col in [
        ("PRIOR", "score_prior"),
        ("LEXICAL_OVERLAP", "score_lexical_overlap"),
    ]:

        credits = []
        margins = []

        for c in contexts:
            sub = [
                r for r in rows
                if r["context_id"] == c
            ]

            best = max(r[col] for r in sub)

            tied = [
                r for r in sub
                if best - r[col] <= 1e-11
            ]

            credit = sum(
                r["preferred"] == "YES" for r in tied
            ) / len(tied)

            correct = next(
                r for r in sub
                if r["preferred"] == "YES"
            )

            best_foil = max(
                r[col] for r in sub
                if r["preferred"] != "YES"
            )

            margin = round(
                correct[col] - best_foil, 9
            )

            credits.append(credit)
            margins.append(margin)

            by_context.append(dict(
                task_type=task,
                language=lang,
                group_id=gid,
                context_id=c,
                baseline=method,
                top1_credit=credit,
                correct_candidate=correct["candidate_id"],
                winner_candidates=";".join(
                    r["candidate_id"] for r in tied
                ),
                preferred_minus_best_foil=margin,
            ))

        by_family.append(dict(
            task_type=task,
            language=lang,
            group_id=gid,
            baseline=method,
            contexts=len(contexts),
            mean_top1_credit=round(
                sum(credits) / len(credits), 6
            ),
            both_contexts_correct=(
                "YES" if all(m > 1e-11 for m in margins)
                else "NO"
            ) if task == "CONTEXT_SWAP_2x2" else "",
        ))

# ----------------------------------------------------
# 6. 汇总统计：以构式族为单位
# ----------------------------------------------------

summary = []

keys = sorted({
    (
        r["task_type"],
        r["language"],
        r["baseline"],
    )
    for r in by_family
})

for task, lang, method in keys:

    subset = [
        r for r in by_family
        if (
            r["task_type"],
            r["language"],
            r["baseline"],
        ) == (task, lang, method)
    ]

    mean_accuracy = sum(
        r["mean_top1_credit"] for r in subset
    ) / len(subset)

    summary.append(dict(
        task_type=task,
        language=lang,
        baseline=method,
        families=len(subset),
        average_family_top1_credit=round(
            mean_accuracy, 6
        ),
        full_2x2_switch_families=(
            sum(
                r["both_contexts_correct"] == "YES"
                for r in subset
            )
            if task == "CONTEXT_SWAP_2x2" else ""
        ),
    ))

# ----------------------------------------------------
# 7. 保存结果：四个独立CSV
# ----------------------------------------------------

def save_csv(filename, rows):
    with (OUT / filename).open(
        "w", newline="", encoding="utf-8-sig"
    ) as f:
        writer = csv.DictWriter(
            f, fieldnames=list(rows[0]),
            extrasaction="ignore"
        )
        writer.writeheader()
        writer.writerows(rows)


save_csv("a2_scored_candidates.csv", stimuli)
save_csv("a2_context_predictions.csv", by_context)
save_csv("a2_family_summary.csv", by_family)
save_csv("a2_summary.csv", summary)

print("\n========== A2 RESULTS ==========")

for r in summary:
    print(
        r["task_type"],
        r["language"],
        r["baseline"],
        "families =", r["families"],
        "accuracy =",
        f"{r['average_family_top1_credit']:.1%}"
    )

print("\n完成！输出目录：", OUT)

请同时上传最终审核 Excel 和 PARSEME ZIP


Saving MWE2027_v2.2_合并_最终修订(2).xlsx to MWE2027_v2.2_合并_最终修订(2).xlsx
Saving parseme_lvc_output.zip to parseme_lvc_output (2).zip
Excel读取成功
总审核记录： 92
A2评分记录： 73
3×3： 45
2×2： 28

PARSEME train句子数：
ZH 1185
EN 101
DE 240

两种词汇基线评分完成

========== A2 RESULTS ==========
CONFIG_3x3 DE LEXICAL_OVERLAP families = 1 accuracy = 33.3%
CONFIG_3x3 DE PRIOR families = 1 accuracy = 33.3%
CONFIG_3x3 EN LEXICAL_OVERLAP families = 2 accuracy = 50.0%
CONFIG_3x3 EN PRIOR families = 2 accuracy = 33.3%
CONFIG_3x3 ZH LEXICAL_OVERLAP families = 2 accuracy = 50.0%
CONFIG_3x3 ZH PRIOR families = 2 accuracy = 33.3%
CONTEXT_SWAP_2x2 DE LEXICAL_OVERLAP families = 2 accuracy = 50.0%
CONTEXT_SWAP_2x2 DE PRIOR families = 2 accuracy = 50.0%
CONTEXT_SWAP_2x2 EN LEXICAL_OVERLAP families = 2 accuracy = 25.0%
CONTEXT_SWAP_2x2 EN PRIOR families = 2 accuracy = 50.0%
CONTEXT_SWAP_2x2 ZH LEXICAL_OVERLAP families = 3 accuracy = 50.0%
CONTEXT_SWAP_2x2 ZH PRIOR families = 3 accuracy = 50.0%

完成！输出目录： /content/a2_results


In [4]:
from google.colab import files

# 四个结果文件分别下载，不打 ZIP 包
files.download("/content/a2_results/a2_summary.csv")
files.download("/content/a2_results/a2_family_summary.csv")
files.download("/content/a2_results/a2_context_predictions.csv")
files.download("/content/a2_results/a2_scored_candidates.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ==========================================================
# MWE 2027 | B-API | Kimi K3.1 Pro + DeepSeek V4 Pro
# Controlled choice: 3x3 configurations + 2x2 role swap
# Input: a2_scored_candidates.csv
# No GPU, no Drive, no extra annotation
# ==========================================================

import re
import csv
import time
import json
import random
import hashlib
from pathlib import Path

import pandas as pd
import requests
from google.colab import files, userdata

# ----------------------------------------------------------
# 0. SETTINGS
# ----------------------------------------------------------

PILOT_ONLY = False
# First run: True  -> 2 contexts per model
# Full run:  False -> all 29 contexts per model

REASONING_EFFORT = "low"   # "low", "high", "max"
MAX_COMPLETION_TOKENS = 2048

MODELS = {
    "KIMI": {
        "model": "kimi-k3.1-pro",
        "url": "https://api.moonshot.cn/v1/chat/completions",
        "secret_names": ["Kimi"]
    },
    "DEEPSEEK": {
        "model": "deepseek-v4-pro",
        "url": "https://api.deepseek.com/chat/completions",
        "secret_names": ["DeepSeek"]
    }
}

OUT = Path("/content/mwe_api_choice_results")
OUT.mkdir(parents=True, exist_ok=True)

RESULT_FILE = OUT / "api_choice_results.csv"

# ----------------------------------------------------------
# 1. LOAD API KEYS FROM COLAB SECRETS
# ----------------------------------------------------------

def load_secret(names):
    for name in names:
        try:
            value = userdata.get(name)
            if value:
                return value
        except Exception:
            pass
    raise RuntimeError(
        "Colab Secret not found. Check: " + ", ".join(names)
    )

KEYS = {
    name: load_secret(config["secret_names"])
    for name, config in MODELS.items()
}

print("Both API keys loaded securely.")

# ----------------------------------------------------------
# 2. LOAD THE SAME 73 A2 SCORING ROWS
# ----------------------------------------------------------

possible_paths = [
    Path("/content/a2_results/a2_scored_candidates.csv"),
    Path("/content/a2_scored_candidates.csv")
]

source = next(
    (p for p in possible_paths if p.exists()), None
)

if source is None:
    print("Upload a2_scored_candidates.csv")
    uploaded = files.upload()
    matches = [
        Path(name) for name in uploaded
        if name.endswith("a2_scored_candidates.csv")
    ]
    assert len(matches) == 1
    source = matches[0]

data = pd.read_csv(
    source, encoding="utf-8-sig",
    dtype={
        "item_id": str,
        "task_type": str,
        "language": str,
        "group_id": str,
        "context_id": str,
        "candidate_id": str,
        "preferred": str
    }
).fillna("")

required = {
    "item_id", "task_type", "language", "group_id",
    "context_id", "candidate_id", "context",
    "completion", "preferred", "score_prior",
    "score_lexical_overlap", "item_validated"
}

assert required.issubset(data.columns)
assert len(data) == 73
assert data["item_id"].is_unique
assert (data["item_validated"] == "YES").all()

print("A2 rows loaded:", len(data))

# ----------------------------------------------------------
# 3. CONSTRUCT CONTEXT-LEVEL MULTIPLE-CHOICE TASKS
# ----------------------------------------------------------

TASK_CANDIDATES = {
    "CONFIG_3x3": ["STATE", "BECOME", "CAUSE"],
    "CONTEXT_SWAP_2x2": ["A", "B"]
}

LETTERS = "ABC"

def lexical_credit(group, column):
    scores = group[column].astype(float)
    best = scores.max()
    tied = group[
        (scores - best).abs() <= 1e-9
    ]
    return (
        (tied["preferred"] == "YES").sum() / len(tied)
    )

contexts = []

group_cols = [
    "task_type", "language", "group_id", "context_id"
]

for key, group in data.groupby(group_cols, sort=True):
    task, lang, family, ctx_id = key

    expected_ids = TASK_CANDIDATES[task]

    assert len(group) == len(expected_ids)
    assert set(group["candidate_id"]) == set(expected_ids)
    assert group["context"].nunique() == 1
    assert (group["preferred"] == "YES").sum() == 1

    candidates = group.to_dict("records")
    candidates.sort(
        key=lambda x: expected_ids.index(x["candidate_id"])
    )

    # Stable randomization: identical options for both APIs
    seed_text = "|".join(key)
    seed = int(
        hashlib.sha256(seed_text.encode()).hexdigest()[:16],
        16
    )
    random.Random(seed).shuffle(candidates)

    options = []
    gold_letter = None

    for i, item in enumerate(candidates):
        letter = LETTERS[i]

        options.append({
            "letter": letter,
            "candidate_id": item["candidate_id"],
            "completion": str(item["completion"]).strip()
        })

        if item["preferred"] == "YES":
            gold_letter = letter

    contexts.append({
        "task_type": task,
        "language": lang,
        "group_id": family,
        "context_id": ctx_id,
        "context": group.iloc[0]["context"],
        "options": options,
        "gold_letter": gold_letter,
        "a2_prior_credit": lexical_credit(
            group, "score_prior"
        ),
        "a2_overlap_credit": lexical_credit(
            group, "score_lexical_overlap"
        )
    })

assert len(contexts) == 29

print("Choice contexts:", len(contexts))
print("Expected complete run: 58 API calls")

# ----------------------------------------------------------
# 4. PROMPT: NO GOLD LABELS SENT TO THE MODEL
# ----------------------------------------------------------

INSTRUCTIONS = {
    "ZH": (
        "阅读下面的语境，选择最自然、最符合语境的一句。"
        "其他句子可能也合语法，但只选最贴切的一句。"
        "只输出选项字母，不要解释。"
    ),
    "EN": (
        "Read the context and select the one sentence "
        "that fits it most naturally. Other options may "
        "also be grammatical. Reply with only the option "
        "letter, without explanation."
    ),
    "DE": (
        "Lies den Kontext und wähle den Satz, der "
        "am natürlichsten dazu passt. Andere Optionen "
        "können ebenfalls grammatisch sein. "
        "Antworte nur mit dem Buchstaben, ohne Erklärung."
    )
}

def make_prompt(item):
    options_text = "\n".join(
        f"{x['letter']}. {x['completion']}"
        for x in item["options"]
    )

    return (
        INSTRUCTIONS[item["language"]]
        + "\n\nKONTEXT / CONTEXT:\n"
        + item["context"]
        + "\n\nOPTIONS:\n"
        + options_text
        + "\n\nAnswer:"
    )

# ----------------------------------------------------------
# 5. API CALL WITH RETRIES
# ----------------------------------------------------------

session = requests.Session()

def call_model(provider, prompt):
    config = MODELS[provider]

    payload = {
        "model": config["model"],
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are completing a controlled "
                    "linguistic preference task. "
                    "Return only one option letter."
                )
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        "reasoning_effort": REASONING_EFFORT
    }

    if provider == "KIMI":
        payload["max_completion_tokens"] = (
            MAX_COMPLETION_TOKENS
        )
    else:
        payload["max_tokens"] = (
            MAX_COMPLETION_TOKENS
        )

    headers = {
        "Authorization": f"Bearer {KEYS[provider]}",
        "Content-Type": "application/json"
    }

    for attempt in range(4):
        try:
            response = session.post(
                config["url"],
                json=payload,
                headers=headers,
                timeout=(20, 240)
            )
        except requests.RequestException:
            if attempt == 3:
                raise
            time.sleep(2 ** attempt + 1)
            continue

        if response.status_code in (
            429, 500, 502, 503, 504
        ):
            if attempt < 3:
                time.sleep(2 ** attempt + 2)
                continue

        if not response.ok:
            raise RuntimeError(
                f"{provider} HTTP {response.status_code}: "
                + response.text[:500]
            )

        result = response.json()
        choice = result["choices"][0]

        content = (
            choice.get("message", {}).get("content") or ""
        )

        if not isinstance(content, str):
            content = str(content)

        usage = result.get("usage") or {}

        return {
            "text": content.strip(),
            "finish_reason": choice.get("finish_reason", ""),
            "actual_model": result.get(
                "model", config["model"]
            ),
            "prompt_tokens": usage.get("prompt_tokens", ""),
            "completion_tokens": usage.get(
                "completion_tokens", ""
            )
        }

    raise RuntimeError("Maximum retries exceeded")

def parse_letter(text, allowed):
    # Strict parsing: no silent guessing from explanations
    match = re.fullmatch(
        r"\s*([ABC])\s*[.)]?\s*",
        text.upper()
    )
    if not match:
        return ""

    value = match.group(1)
    return value if value in allowed else ""

# ----------------------------------------------------------
# 6. RUN BOTH FLAGSHIP MODELS
# ----------------------------------------------------------

def job_key(provider, item):
    return "|".join([
        provider,
        item["task_type"],
        item["language"],
        item["group_id"],
        item["context_id"]
    ])

if RESULT_FILE.exists():
    existing = pd.read_csv(
        RESULT_FILE, dtype=str
    ).fillna("").to_dict("records")
else:
    existing = []

results_by_key = {
    row["job_key"]: row for row in existing
}

selected_contexts = (
    contexts[:2] if PILOT_ONLY else contexts
)

jobs = [
    (provider, item)
    for provider in MODELS
    for item in selected_contexts
]

for index, (provider, item) in enumerate(jobs, 1):
    key = job_key(provider, item)

    # Reuse only successful previous responses
    if (
        key in results_by_key
        and results_by_key[key].get("status") == "OK"
    ):
        print("Cached:", key)
        continue

    print(
        f"[{index}/{len(jobs)}]",
        provider,
        item["language"],
        item["group_id"],
        item["context_id"]
    )

    try:
        reply = call_model(
            provider, make_prompt(item)
        )

        allowed = [
            x["letter"] for x in item["options"]
        ]

        predicted = parse_letter(
            reply["text"], allowed
        )

        status = "OK" if predicted else "INVALID_OUTPUT"
        correct = (
            "1" if predicted == item["gold_letter"] else "0"
        ) if predicted else ""

        row = {
            "job_key": key,
            "provider": provider,
            "requested_model": MODELS[provider]["model"],
            "actual_model": reply["actual_model"],
            "task_type": item["task_type"],
            "language": item["language"],
            "group_id": item["group_id"],
            "context_id": item["context_id"],
            "gold_letter": item["gold_letter"],
            "predicted_letter": predicted,
            "is_correct": correct,
            "status": status,
            "raw_output": reply["text"][:500],
            "finish_reason": reply["finish_reason"],
            "a2_prior_credit": item["a2_prior_credit"],
            "a2_overlap_credit": item["a2_overlap_credit"],
            "options_json": json.dumps(
                item["options"], ensure_ascii=False
            ),
            "prompt_tokens": reply["prompt_tokens"],
            "completion_tokens": reply["completion_tokens"]
        }

    except Exception as error:
        row = {
            "job_key": key,
            "provider": provider,
            "requested_model": MODELS[provider]["model"],
            "actual_model": "",
            "task_type": item["task_type"],
            "language": item["language"],
            "group_id": item["group_id"],
            "context_id": item["context_id"],
            "gold_letter": item["gold_letter"],
            "predicted_letter": "",
            "is_correct": "",
            "status": "API_ERROR",
            "raw_output": str(error)[:500],
            "finish_reason": "",
            "a2_prior_credit": item["a2_prior_credit"],
            "a2_overlap_credit": item["a2_overlap_credit"],
            "options_json": json.dumps(
                item["options"], ensure_ascii=False
            ),
            "prompt_tokens": "",
            "completion_tokens": ""
        }

    results_by_key[key] = row

    # Checkpoint after every request
    pd.DataFrame(
        list(results_by_key.values())
    ).to_csv(
        RESULT_FILE,
        index=False,
        encoding="utf-8-sig"
    )

    print(
        "Status:", row["status"],
        "Predicted:", row["predicted_letter"],
        "Gold:", row["gold_letter"]
    )

# ----------------------------------------------------------
# 7. SUMMARY: KIMI vs DEEPSEEK vs A2
# ----------------------------------------------------------

result_df = pd.read_csv(
    RESULT_FILE, dtype=str
).fillna("")

summary = []

for (provider, task, lang), sub in result_df.groupby(
    ["provider", "task_type", "language"]
):
    valid = sub[sub["status"] == "OK"].copy()

    if len(valid):
        accuracy = valid["is_correct"].astype(float).mean()
        a2_prior = valid[
            "a2_prior_credit"
        ].astype(float).mean()
        a2_overlap = valid[
            "a2_overlap_credit"
        ].astype(float).mean()
    else:
        accuracy = None
        a2_prior = None
        a2_overlap = None

    summary.append({
        "provider": provider,
        "task_type": task,
        "language": lang,
        "contexts_attempted": len(sub),
        "valid_responses": len(valid),
        "api_accuracy": accuracy,
        "a2_prior_same_contexts": a2_prior,
        "a2_overlap_same_contexts": a2_overlap
    })

summary_df = pd.DataFrame(summary)

summary_file = OUT / "api_choice_summary.csv"
summary_df.to_csv(
    summary_file,
    index=False,
    encoding="utf-8-sig"
)

# Family-level evaluation
family_summary = []

for key, sub in result_df.groupby(
    ["provider", "task_type", "language", "group_id"]
):
    provider, task, lang, family = key
    expected_n = 3 if task == "CONFIG_3x3" else 2
    valid = sub[sub["status"] == "OK"]

    complete = len(valid) == expected_n

    family_summary.append({
        "provider": provider,
        "task_type": task,
        "language": lang,
        "group_id": family,
        "expected_contexts": expected_n,
        "valid_contexts": len(valid),
        "family_complete": complete,
        "family_accuracy": (
            valid["is_correct"].astype(float).mean()
            if len(valid) else None
        ),
        "all_contexts_correct": (
            bool(
                (valid["is_correct"] == "1").all()
            )
            if complete else None
        )
    })

family_df = pd.DataFrame(family_summary)
family_file = OUT / "api_choice_family_summary.csv"

family_df.to_csv(
    family_file,
    index=False,
    encoding="utf-8-sig"
)

metadata = {
    "experiment": "API controlled forced-choice",
    "not_conditional_surprisal": True,
    "kimi_model": MODELS["KIMI"]["model"],
    "deepseek_model": MODELS["DEEPSEEK"]["model"],
    "reasoning_effort": REASONING_EFFORT,
    "pilot_only": PILOT_ONLY,
    "full_context_count_per_model": len(contexts),
    "input": str(source),
    "candidate_order": "deterministic shuffled",
    "prompt_runs_per_context": 1,
    "experimental_status": "exploratory"
}

(OUT / "api_choice_metadata.json").write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("\n========== RESULTS ==========")
print(summary_df.to_string(index=False))

print("\nAll outputs saved in:", OUT)


Both API keys loaded securely.
A2 rows loaded: 73
Choice contexts: 29
Expected complete run: 58 API calls
Cached: KIMI|CONFIG_3x3|DE|DE_T01|BECOME
Cached: KIMI|CONFIG_3x3|DE|DE_T01|CAUSE
[3/58] KIMI DE DE_T01 STATE
Status: OK Predicted: B Gold: B
[4/58] KIMI EN EN_T01 BECOME
Status: OK Predicted: C Gold: C
[5/58] KIMI EN EN_T01 CAUSE
Status: OK Predicted: B Gold: B
[6/58] KIMI EN EN_T01 STATE
Status: OK Predicted: A Gold: A
[7/58] KIMI EN EN_T02 BECOME
Status: OK Predicted: A Gold: A
[8/58] KIMI EN EN_T02 CAUSE
Status: OK Predicted: A Gold: A
[9/58] KIMI EN EN_T02 STATE
Status: OK Predicted: C Gold: C
[10/58] KIMI ZH ZH_T01 BECOME
Status: OK Predicted: A Gold: A
[11/58] KIMI ZH ZH_T01 CAUSE
Status: OK Predicted: B Gold: B
[12/58] KIMI ZH ZH_T01 STATE
Status: OK Predicted: B Gold: B
[13/58] KIMI ZH ZH_T02 BECOME
Status: OK Predicted: C Gold: C
[14/58] KIMI ZH ZH_T02 CAUSE
Status: OK Predicted: A Gold: A
[15/58] KIMI ZH ZH_T02 STATE
Status: OK Predicted: C Gold: C
[16/58] KIMI DE DE_DRUC

LLM Exp. Conditional Surprisal

In [5]:

# ==================================================
# MWE 2027 | v2.1 | B: LM Conditional Surprisal
# 73 reviewed candidates | 12 construction families
# One frozen multilingual causal LM
# ==================================================


from pathlib import Path
from google.colab import files
import re

def find_a2_csv():
    # 1. 优先读取 A2 默认输出
    standard = Path("/content/a2_results/a2_scored_candidates.csv")
    if standard.is_file():
        return standard

    # 2. 自动查找 /content/ 中上传的文件
    # 支持 a2_scored_candidates.csv
    # 以及 a2_scored_candidates (1).csv、(2).csv 等
    candidates = [
        p for p in Path("/content").glob("a2_scored_candidates*.csv")
        if re.fullmatch(
            r"a2_scored_candidates(?: \(\d+\))?\.csv",
            p.name
        )
    ]

    if candidates:
        # 同名文件有多个版本时，优先最新修改的文件
        return max(candidates, key=lambda p: p.stat().st_mtime)

    # 3. 如果确实没有文件，才请求上传
    print("未找到 A2 CSV，请上传 a2_scored_candidates.csv")
    uploaded = files.upload()

    candidates = [Path("/content") / name for name in uploaded]

    candidates = [
        p for p in candidates
        if p.is_file() and re.fullmatch(
            r"a2_scored_candidates(?: \(\d+\))?\.csv",
            p.name
        )
    ]

    if not candidates:
        raise FileNotFoundError("没有找到有效的 A2 CSV 文件")

    return max(candidates, key=lambda p: p.stat().st_mtime)


input_path = find_a2_csv()
print("自动使用文件：", input_path)


with input_path.open(encoding="utf-8-sig", newline="") as f:
    rows = list(csv.DictReader(f))

required = {
    "item_id", "task_type", "language", "group_id",
    "context_id", "candidate_id", "context",
    "completion", "preferred", "item_validated",
    "score_prior", "score_lexical_overlap"
}

assert rows and required <= set(rows[0])
assert len(rows) == 73
assert len({r["item_id"] for r in rows}) == 73
assert all(r["item_validated"] == "YES" for r in rows)

# 2. Check the experiment grids
TASKS = {
    "CONFIG_3x3": (
        ("STATE", "BECOME", "CAUSE"),
        ("STATE", "BECOME", "CAUSE")
    ),
    "CONTEXT_SWAP_2x2": (
        ("CA", "CB"),
        ("A", "B")
    )
}

groups = defaultdict(list)

for r in rows:
    assert r["task_type"] in TASKS
    assert r["language"] in ("ZH", "EN", "DE")
    groups[
        (r["task_type"], r["language"], r["group_id"])
    ].append(r)

for key, group in groups.items():
    contexts, candidates = TASKS[key[0]]

    expected = {
        (c, k) for c in contexts for k in candidates
    }

    observed = [
        (r["context_id"], r["candidate_id"])
        for r in group
    ]

    assert len(observed) == len(expected)
    assert set(observed) == expected

    for c in contexts:
        sub = [r for r in group if r["context_id"] == c]
        assert len({r["context"] for r in sub}) == 1
        assert sum(r["preferred"] == "YES" for r in sub) == 1

    for k in candidates:
        assert len({
            r["completion"] for r in group
            if r["candidate_id"] == k
        }) == 1

print("GRID CHECK PASSED")
print("Scoring rows:", len(rows))
print("Construction families:", len(groups))

# 3. Load frozen base model
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM

assert torch.cuda.is_available(), (
    "Please enable T4 GPU in Colab Runtime settings."
)

DEVICE = "cuda"
MODEL_ID = "Qwen/Qwen2.5-1.5B"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, use_fast=True
)
assert tokenizer.is_fast

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16
).to(DEVICE).eval()

model.config.use_cache = False

print("Model loaded:", MODEL_ID)
print("GPU:", torch.cuda.get_device_name(0))

# 4. Conditional target-token scoring
NEUTRAL = {
    "ZH": "下面是一句话。\n\n",
    "EN": "Here is a sentence.\n\n",
    "DE": "Hier ist ein Satz.\n\n"
}

cache = {}

def score_completion(prefix, completion):
    key = (prefix, completion)
    if key in cache:
        return cache[key]

    text = prefix + completion

    encoded = tokenizer(
        text,
        add_special_tokens=True,
        return_offsets_mapping=True
    )

    ids = encoded["input_ids"]
    offsets = encoded["offset_mapping"]
    boundary = len(prefix)

    target_positions = [
        i for i, (a, b) in enumerate(offsets)
        if b > boundary
    ]

    assert target_positions
    assert target_positions[0] > 0

    # Ensure no token straddles the target boundary.
    crossing = [
        (i, a, b) for i, (a, b) in enumerate(offsets)
        if b > boundary and a < boundary
    ]
    if crossing:
        raise ValueError(
            f"Tokenizer crosses target boundary: {crossing}. "
            "Please report this error before interpreting scores."
        )

    assert target_positions == list(range(
        target_positions[0], target_positions[-1] + 1
    ))

    tensor = torch.tensor(
        [ids], dtype=torch.long, device=DEVICE
    )

    with torch.inference_mode():
        logits = model(
            input_ids=tensor, use_cache=False
        ).logits[0]

        predicted_logits = logits[
            [i - 1 for i in target_positions], :
        ].float()

        target_ids = tensor[0, target_positions]

        token_logps = torch.log_softmax(
            predicted_logits, dim=-1
        ).gather(
            -1, target_ids[:, None]
        ).squeeze(-1)

        total = float(token_logps.sum().item())

    n = len(target_positions)
    result = (total, total / n, n)
    cache[key] = result
    return result

# 5. Run the actual LM on all 73 rows
for i, r in enumerate(rows, 1):
    completion = r["completion"].strip()
    context_prefix = r["context"].rstrip() + "\n\n"

    ct, cm, n = score_completion(
        context_prefix, completion
    )

    nt, nm, _ = score_completion(
        NEUTRAL[r["language"]], completion
    )

    r["lm_cond_total"] = ct
    r["lm_cond_mean"] = cm
    r["lm_neutral_mean"] = nm
    r["lm_context_lift"] = cm - nm
    r["lm_target_tokens"] = n
    r["model_id"] = MODEL_ID

    if i % 10 == 0 or i == len(rows):
        print(f"LM scored {i}/{len(rows)}")

# 6. Compare A2 and B on identical grids
METHODS = {
    "A2_PRIOR": "score_prior",
    "A2_TFIDF": "score_lexical_overlap",
    "B_LM_COND": "lm_cond_mean",
    "B_LM_CONTEXT_LIFT": "lm_context_lift"
}

context_results = []
family_results = []

for (task, lang, gid), group in sorted(groups.items()):
    contexts, candidates = TASKS[task]

    for method, column in METHODS.items():
        credits = []
        margins = []

        for ctx in contexts:
            sub = [
                r for r in group
                if r["context_id"] == ctx
            ]

            scores = [float(r[column]) for r in sub]
            best = max(scores)

            winners = [
                r for r, score in zip(sub, scores)
                if abs(score - best) <= 1e-9
            ]

            credit = sum(
                r["preferred"] == "YES" for r in winners
            ) / len(winners)

            correct = next(
                r for r in sub
                if r["preferred"] == "YES"
            )

            foil = max(
                float(r[column]) for r in sub
                if r["preferred"] == "NO"
            )

            margin = float(correct[column]) - foil

            credits.append(credit)
            margins.append(margin)

            context_results.append({
                "task_type": task,
                "language": lang,
                "group_id": gid,
                "context_id": ctx,
                "method": method,
                "top1_credit": credit,
                "correct_candidate": correct["candidate_id"],
                "winner_candidates": ";".join(
                    r["candidate_id"] for r in winners
                ),
                "preferred_margin": margin
            })

        family_results.append({
            "task_type": task,
            "language": lang,
            "group_id": gid,
            "method": method,
            "contexts": len(contexts),
            "mean_top1_credit": sum(credits) / len(credits),
            "all_contexts_correct": int(
                all(m > 1e-9 for m in margins)
            ),
            "sum_preferred_margins": sum(margins)
        })

# 7. Summary by language, task and method
summary = []

for task in TASKS:
    for lang in ("ZH", "EN", "DE"):
        for method in METHODS:

            subset = [
                r for r in family_results
                if (
                    r["task_type"],
                    r["language"],
                    r["method"]
                ) == (task, lang, method)
            ]

            if not subset:
                continue

            summary.append({
                "task_type": task,
                "language": lang,
                "method": method,
                "families": len(subset),
                "average_family_top1_credit": sum(
                    float(r["mean_top1_credit"])
                    for r in subset
                ) / len(subset),
                "all_contexts_correct_families": sum(
                    int(r["all_contexts_correct"])
                    for r in subset
                )
            })

# 8. Export results
OUT = Path("/content/b_lm_results")
OUT.mkdir(exist_ok=True)

def save_csv(name, data):
    with (OUT / name).open(
        "w", newline="", encoding="utf-8-sig"
    ) as f:
        writer = csv.DictWriter(
            f, fieldnames=list(data[0])
        )
        writer.writeheader()
        writer.writerows(data)

save_csv("b_scored_candidates.csv", rows)
save_csv("b_context_comparison.csv", context_results)
save_csv("b_family_comparison.csv", family_results)
save_csv("b_summary.csv", summary)

metadata = {
    "model_id": MODEL_ID,
    "model_commit": getattr(
        model.config, "_commit_hash", None
    ),
    "transformers_version": transformers.__version__,
    "torch_version": torch.__version__,
    "stimulus_rows": len(rows),
    "families": len(groups),
    "scoring": "target-only conditional log-probability",
    "analysis_status": "exploratory pilot"
}

(OUT / "b_run_metadata.json").write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("\n========== B RESULTS ==========")
for r in summary:
    print(
        r["task_type"], r["language"], r["method"],
        "families =", r["families"],
        "accuracy =",
        f"{r['average_family_top1_credit']:.1%}"
    )

print("\nResults saved to:", OUT)


自动使用文件： /content/a2_scored_candidates (1).csv
GRID CHECK PASSED
Scoring rows: 73
Construction families: 12


config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (2 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Model loaded: Qwen/Qwen2.5-1.5B
GPU: Tesla T4
LM scored 10/73
LM scored 20/73
LM scored 30/73
LM scored 40/73
LM scored 50/73
LM scored 60/73
LM scored 70/73
LM scored 73/73

========== B RESULTS ==========
CONFIG_3x3 ZH A2_PRIOR families = 2 accuracy = 33.3%
CONFIG_3x3 ZH A2_TFIDF families = 2 accuracy = 50.0%
CONFIG_3x3 ZH B_LM_COND families = 2 accuracy = 66.7%
CONFIG_3x3 ZH B_LM_CONTEXT_LIFT families = 2 accuracy = 83.3%
CONFIG_3x3 EN A2_PRIOR families = 2 accuracy = 33.3%
CONFIG_3x3 EN A2_TFIDF families = 2 accuracy = 50.0%
CONFIG_3x3 EN B_LM_COND families = 2 accuracy = 66.7%
CONFIG_3x3 EN B_LM_CONTEXT_LIFT families = 2 accuracy = 66.7%
CONFIG_3x3 DE A2_PRIOR families = 1 accuracy = 33.3%
CONFIG_3x3 DE A2_TFIDF families = 1 accuracy = 33.3%
CONFIG_3x3 DE B_LM_COND families = 1 accuracy = 33.3%
CONFIG_3x3 DE B_LM_CONTEXT_LIFT families = 1 accuracy = 66.7%
CONTEXT_SWAP_2x2 ZH A2_PRIOR families = 3 accuracy = 50.0%
CONTEXT_SWAP_2x2 ZH A2_TFIDF families = 3 accuracy = 50.0%
CONTEXT_S

In [6]:

# B1 Error Audit | MWE 2027
import pandas as pd
from pathlib import Path
from google.colab import files

source = Path(
    "/content/b_lm_results/b_scored_candidates.csv"
)
assert source.exists(), "请先确认 B1 已运行完成"

df = pd.read_csv(source, encoding="utf-8-sig")
assert len(df) == 73

methods = {
    "A2_PRIOR": "score_prior",
    "A2_TFIDF": "score_lexical_overlap",
    "B1_COND": "lm_cond_mean",
    "B1_CONTEXT_LIFT": "lm_context_lift"
}

group_cols = [
    "task_type", "language",
    "group_id", "context_id"
]

results = []

for key, group in df.groupby(group_cols):
    gold = group.loc[
        group["preferred"] == "YES"
    ].iloc[0]

    record = dict(zip(group_cols, key))
    record["gold_candidate"] = gold["candidate_id"]
    record["context"] = gold["context"]

    for method, column in methods.items():
        scores = group[column].astype(float)
        maximum = scores.max()

        winners = group.loc[
            (scores - maximum).abs() <= 1e-9,
            "candidate_id"
        ].tolist()

        gold_score = float(gold[column])
        record[f"{method}_winner"] = ";".join(winners)
        record[f"{method}_correct"] = (
            len(winners) == 1
            and winners[0] == gold["candidate_id"]
        )
        record[f"{method}_margin"] = (
            gold_score - maximum
        )

    results.append(record)

audit = pd.DataFrame(results)

wrong = audit.loc[
    ~audit["B1_COND_correct"]
    | ~audit["B1_CONTEXT_LIFT_correct"]
].copy()

out = Path("/content/b_lm_results")
audit.to_csv(
    out / "b1_a2_context_audit.csv",
    index=False, encoding="utf-8-sig"
)
wrong.to_csv(
    out / "b1_error_cases.csv",
    index=False, encoding="utf-8-sig"
)

print("Total contexts:", len(audit))
print("Cases with B1 errors or ties:", len(wrong))

display(wrong[
    [
        "task_type", "language", "group_id",
        "context_id", "gold_candidate",
        "B1_COND_winner",
        "B1_CONTEXT_LIFT_winner"
    ]
])

files.download(str(out / "b1_error_cases.csv"))
files.download(str(out / "b1_a2_context_audit.csv"))

Total contexts: 29
Cases with B1 errors or ties: 13


,task_type,language,group_id,context_id,gold_candidate,B1_COND_winner,B1_CONTEXT_LIFT_winner
0,CONFIG_3x3,DE,DE_T01,BECOME,BECOME,CAUSE,STATE
2,CONFIG_3x3,DE,DE_T01,STATE,STATE,CAUSE,STATE
3,CONFIG_3x3,EN,EN_T01,BECOME,BECOME,STATE,BECOME
4,CONFIG_3x3,EN,EN_T01,CAUSE,CAUSE,STATE,BECOME
5,CONFIG_3x3,EN,EN_T01,STATE,STATE,STATE,BECOME
11,CONFIG_3x3,ZH,ZH_T01,STATE,STATE,BECOME,STATE
12,CONFIG_3x3,ZH,ZH_T02,BECOME,BECOME,CAUSE,CAUSE
15,CONTEXT_SWAP_2x2,DE,DE_DRUCK,CA,A,B,B
16,CONTEXT_SWAP_2x2,DE,DE_DRUCK,CB,B,A,A
18,CONTEXT_SWAP_2x2,DE,DE_KONTROLLE,CB,B,A,B


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [7]:
print("Qwen loaded:", "model" in globals())
print("Dataset loaded:", "rows" in globals())
print("GPU available:", __import__("torch").cuda.is_available())

Qwen loaded: True
Dataset loaded: True
GPU available: True


In [8]:

# ========================================================
# MWE 2027 | v2.1 | B2 Hidden-State Linear Probe
# Model: Qwen/Qwen2.5-1.5B (Base)
# 73 candidates | 12 construction families
# Frozen hidden states + leave-one-family-out evaluation
# ========================================================

from pathlib import Path
from collections import defaultdict
from google.colab import files

import re
import json
import csv
import numpy as np
import pandas as pd
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.preprocessing import normalize

# --------------------------------------------------------
# 1. SETTINGS
# --------------------------------------------------------

MODEL_ID = "Qwen/Qwen2.5-1.5B"
OUT = Path("/content/b2_hidden_probe_results")
OUT.mkdir(parents=True, exist_ok=True)

assert torch.cuda.is_available(), (
    "请在 Colab 中启用 T4 GPU：Runtime -> Change runtime type"
)

print("GPU:", torch.cuda.get_device_name(0))


# --------------------------------------------------------
# 2. AUTOMATICALLY FIND INPUT CSV
# --------------------------------------------------------

def find_input():

    # Prefer B1 output because it also contains LM scores.
    direct_paths = [
        Path("/content/b_lm_results/b_scored_candidates.csv"),
        Path("/content/a2_results/a2_scored_candidates.csv"),
    ]

    for p in direct_paths:
        if p.is_file():
            return p

    # Search for uploaded copies, including (1), (2), etc.
    patterns = [
        r"b_scored_candidates(?: \(\d+\))?\.csv",
        r"a2_scored_candidates(?: \(\d+\))?\.csv"
    ]

    for pattern in patterns:
        candidates = [
            p for p in Path("/content").glob("*.csv")
            if re.fullmatch(pattern, p.name)
        ]

        if candidates:
            return max(
                candidates,
                key=lambda p: p.stat().st_mtime
            )

    return None


source = find_input()

if source is None:
    print("没有找到CSV，请上传B1或A2的scored_candidates文件")
    files.upload()
    source = find_input()

if source is None:
    raise FileNotFoundError("没有找到有效的实验输入CSV")

print("Automatically using:", source)

df = pd.read_csv(
    source,
    encoding="utf-8-sig",
    dtype=str,
    keep_default_na=False
)

required = {
    "item_id", "task_type", "language",
    "group_id", "context_id", "candidate_id",
    "context", "completion", "preferred",
    "item_validated"
}

assert required.issubset(df.columns), (
    f"Missing columns: {required - set(df.columns)}"
)

assert len(df) == 73
assert df["item_id"].is_unique
assert (df["item_validated"] == "YES").all()
assert df["preferred"].isin(["YES", "NO"]).all()

TASKS = {
    "CONFIG_3x3": (
        ["STATE", "BECOME", "CAUSE"],
        ["STATE", "BECOME", "CAUSE"]
    ),
    "CONTEXT_SWAP_2x2": (
        ["CA", "CB"],
        ["A", "B"]
    )
}

assert df["task_type"].isin(TASKS.keys()).all()
assert df["language"].isin(["ZH", "EN", "DE"]).all()

# Check grids
for (task, lang, family), g in df.groupby(
    ["task_type", "language", "group_id"]
):
    contexts, candidates = TASKS[task]

    expected = {
        (c, k)
        for c in contexts
        for k in candidates
    }

    observed = set(zip(
        g["context_id"], g["candidate_id"]
    ))

    assert observed == expected
    assert len(g) == len(expected)

    for ctx, sub in g.groupby("context_id"):
        assert sub["context"].nunique() == 1
        assert (sub["preferred"] == "YES").sum() == 1

    for candidate, sub in g.groupby("candidate_id"):
        assert sub["completion"].nunique() == 1

print("Data check passed:", len(df), "rows")


# --------------------------------------------------------
# 3. REUSE OR LOAD THE FROZEN QWEN MODEL
# --------------------------------------------------------

reuse_model = False

if "model" in globals() and "tokenizer" in globals():
    current_model_id = getattr(
        model.config, "_name_or_path", ""
    )

    if current_model_id == MODEL_ID and tokenizer.is_fast:
        reuse_model = True

if reuse_model:
    print("Reusing Qwen from previous B1 session")

    model = model.to("cuda")

else:
    print("Loading Qwen model...")

    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_ID,
        use_fast=True
    )

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        dtype=torch.float16
    ).to("cuda")

model.eval()
model.config.use_cache = False
model.requires_grad_(False)

assert tokenizer.is_fast

DEVICE = next(model.parameters()).device

N_LAYERS = model.config.num_hidden_layers

# Fixed middle layer: not selected using test results
LAYER_INDEX = N_LAYERS // 2

print("Model:", MODEL_ID)
print("Transformer layers:", N_LAYERS)
print("Hidden-state index:", LAYER_INDEX)
print("Hidden size:", model.config.hidden_size)


# --------------------------------------------------------
# 4. EXTRACT TARGET-TOKEN HIDDEN REPRESENTATIONS
# --------------------------------------------------------

NEUTRAL = {
    "ZH": "下面是一句话。\n\n",
    "EN": "Here is a sentence.\n\n",
    "DE": "Hier ist ein Satz.\n\n"
}

def extract_hidden(prefix, completion):

    full_text = prefix + completion
    boundary = len(prefix)

    encoded = tokenizer(
        full_text,
        return_tensors="pt",
        return_offsets_mapping=True,
        add_special_tokens=True
    )

    offsets = encoded.pop(
        "offset_mapping"
    )[0].tolist()

    # Check context / completion token boundary
    crossing = [
        (i, a, b)
        for i, (a, b) in enumerate(offsets)
        if a < boundary < b
    ]

    if crossing:
        raise ValueError(
            f"Token crosses target boundary: {crossing}"
        )

    target_positions = [
        i
        for i, (a, b) in enumerate(offsets)
        if b > boundary
    ]

    assert target_positions
    assert target_positions[0] > 0

    inputs = {
        k: v.to(DEVICE)
        for k, v in encoded.items()
    }

    with torch.inference_mode():

        # Use transformer backbone, avoiding LM logits
        outputs = model.model(
            **inputs,
            output_hidden_states=True,
            use_cache=False,
            return_dict=True
        )

        hidden = outputs.hidden_states[LAYER_INDEX]

        vector = (
            hidden[0, target_positions, :]
            .float()
            .mean(dim=0)
            .cpu()
            .numpy()
        )

    return vector


# --------------------------------------------------------
# 5. COMPUTE CONTEXT - NEUTRAL DIFFERENCE
# --------------------------------------------------------

# Cache neutral representations for repeated candidates
neutral_cache = {}
features = []

records = df.to_dict("records")

for i, r in enumerate(records, start=1):

    lang = r["language"]
    completion = r["completion"].strip()

    context_prefix = (
        r["context"].rstrip() + "\n\n"
    )

    h_context = extract_hidden(
        context_prefix,
        completion
    )

    neutral_key = (lang, completion)

    if neutral_key not in neutral_cache:
        neutral_cache[neutral_key] = extract_hidden(
            NEUTRAL[lang],
            completion
        )

    h_neutral = neutral_cache[neutral_key]

    # Contextual difference representation
    delta = h_context - h_neutral

    features.append(
        delta.astype(np.float32)
    )

    if i % 10 == 0 or i == len(records):
        print(f"Extracted {i}/{len(records)}")


X_hidden = np.stack(features)

assert X_hidden.shape == (
    73, model.config.hidden_size
)
assert np.isfinite(X_hidden).all()

# Save vectors for reproducibility
np.save(
    OUT / "b2_hidden_delta.npy",
    X_hidden
)

print("\nHidden-state extraction complete")
print("Feature matrix:", X_hidden.shape)


# --------------------------------------------------------
# 6. LEAVE-ONE-FAMILY-OUT LINEAR PROBE
# --------------------------------------------------------

# L2-normalize each vector independently
X = normalize(X_hidden, norm="l2")

# Classification target:
# Is this candidate preferred in this context?
y = np.array([
    int(r["preferred"] == "YES")
    for r in records
])

task_labels = np.array([
    r["task_type"] for r in records
])

# One full construction family must remain together.
family_labels = np.array([
    "::".join([
        r["task_type"],
        r["language"],
        r["group_id"]
    ])
    for r in records
])

oof_probabilities = np.full(
    len(records), np.nan
)

fold_records = []

for task in sorted(set(task_labels)):

    indices = np.where(
        task_labels == task
    )[0]

    X_task = X[indices]
    y_task = y[indices]
    family_task = family_labels[indices]

    logo = LeaveOneGroupOut()

    print("\nTASK:", task)
    print("Families:", len(set(family_task)))

    for train_idx, test_idx in logo.split(
        X_task,
        y_task,
        family_task
    ):

        assert len(np.unique(y_task[train_idx])) == 2

        clf = LogisticRegression(
            penalty="l2",
            C=1.0,
            class_weight="balanced",
            solver="liblinear",
            max_iter=2000,
            random_state=42
        )

        clf.fit(
            X_task[train_idx],
            y_task[train_idx]
        )

        positive_column = list(
            clf.classes_
        ).index(1)

        probability = clf.predict_proba(
            X_task[test_idx]
        )[:, positive_column]

        original_indices = indices[test_idx]

        oof_probabilities[
            original_indices
        ] = probability

        heldout_family = family_task[
            test_idx
        ][0]

        train_families = set(
            family_task[train_idx]
        )

        assert heldout_family not in train_families

        fold_records.append({
            "task_type": task,
            "heldout_family": heldout_family,
            "train_rows": len(train_idx),
            "test_rows": len(test_idx),
            "train_families": len(train_families),
            "train_positive": int(
                y_task[train_idx].sum()
            ),
            "test_positive": int(
                y_task[test_idx].sum()
            ),
            "heldout_not_in_train": "YES"
        })

        print("Held out:", heldout_family)


assert np.isfinite(oof_probabilities).all()

df["probe_probability"] = oof_probabilities
df["probe_gold"] = y
df["probe_layer"] = LAYER_INDEX
df["probe_split"] = "LOFO"
df["probe_model"] = MODEL_ID

print("\nCross-validated predictions complete")


# --------------------------------------------------------
# 7. EVALUATE CONTEXT-LEVEL TOP-1 ACCURACY
# --------------------------------------------------------

context_results = []

for key, g in df.groupby([
    "task_type", "language",
    "group_id", "context_id"
]):

    task, lang, family, context_id = key

    scores = g["probe_probability"].to_numpy(
        dtype=float
    )

    maximum = scores.max()

    tied = g.loc[
        np.isclose(
            scores,
            maximum,
            rtol=0,
            atol=1e-9
        )
    ]

    correct = g.loc[
        g["preferred"] == "YES"
    ].iloc[0]

    best_foil = g.loc[
        g["preferred"] == "NO",
        "probe_probability"
    ].max()

    margin = (
        float(correct["probe_probability"])
        - float(best_foil)
    )

    credit = (
        (tied["preferred"] == "YES").sum()
        / len(tied)
    )

    context_results.append({
        "task_type": task,
        "language": lang,
        "group_id": family,
        "context_id": context_id,
        "method": "B2_HIDDEN_LOFO",
        "correct_candidate": correct["candidate_id"],
        "winner_candidates": ";".join(
            tied["candidate_id"]
        ),
        "top1_credit": float(credit),
        "preferred_margin": float(margin)
    })


# --------------------------------------------------------
# 8. FAMILY-LEVEL EVALUATION
# --------------------------------------------------------

contexts_df = pd.DataFrame(context_results)
family_results = []

for key, g in contexts_df.groupby([
    "task_type", "language", "group_id"
]):

    task, lang, family = key

    family_results.append({
        "task_type": task,
        "language": lang,
        "group_id": family,
        "method": "B2_HIDDEN_LOFO",
        "contexts": len(g),
        "mean_top1_credit": g[
            "top1_credit"
        ].mean(),
        "all_contexts_correct": int(
            (g["preferred_margin"] > 1e-9).all()
        ),
        "sum_preferred_margins": g[
            "preferred_margin"
        ].sum()
    })


family_df = pd.DataFrame(family_results)

# --------------------------------------------------------
# 9. LANGUAGE-LEVEL SUMMARY
# --------------------------------------------------------

summary_results = []

for key, g in family_df.groupby([
    "task_type", "language", "method"
]):

    task, lang, method = key

    summary_results.append({
        "task_type": task,
        "language": lang,
        "method": method,
        "families": len(g),
        "average_family_top1_credit": g[
            "mean_top1_credit"
        ].mean(),
        "all_contexts_correct_families": int(
            g["all_contexts_correct"].sum()
        )
    })


summary_df = pd.DataFrame(summary_results)


# --------------------------------------------------------
# 10. SAVE ALL RESULTS
# --------------------------------------------------------

df.to_csv(
    OUT / "b2_scored_candidates.csv",
    index=False, encoding="utf-8-sig"
)

contexts_df.to_csv(
    OUT / "b2_context_comparison.csv",
    index=False, encoding="utf-8-sig"
)

family_df.to_csv(
    OUT / "b2_family_comparison.csv",
    index=False, encoding="utf-8-sig"
)

summary_df.to_csv(
    OUT / "b2_summary.csv",
    index=False, encoding="utf-8-sig"
)

pd.DataFrame(fold_records).to_csv(
    OUT / "b2_folds.csv",
    index=False, encoding="utf-8-sig"
)

metadata = {
    "model_id": MODEL_ID,
    "model_commit": getattr(
        model.config, "_commit_hash", None
    ),
    "layer_index": LAYER_INDEX,
    "hidden_size": int(model.config.hidden_size),
    "number_of_items": len(df),
    "number_of_families": len(fold_records),
    "input_file": str(source),
    "target": "context-completion compatibility",
    "features": "context_hidden - neutral_hidden",
    "pooling": "mean completion-token hidden state",
    "classifier": "L2 LogisticRegression, C=1.0",
    "evaluation": "leave-one-family-out",
    "interpretation": "exploratory linear decodability"
}

(OUT / "b2_metadata.json").write_text(
    json.dumps(
        metadata,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print("\n========== B2 RESULTS ==========")

for _, r in summary_df.iterrows():
    print(
        r["task_type"],
        r["language"],
        "families =", r["families"],
        "accuracy =",
        f"{r['average_family_top1_credit']:.1%}",
        "fully correct families =",
        r["all_contexts_correct_families"]
    )

print("\nFinished. Results in:", OUT)


GPU: Tesla T4
Automatically using: /content/b_lm_results/b_scored_candidates.csv
Data check passed: 73 rows
Reusing Qwen from previous B1 session
Model: Qwen/Qwen2.5-1.5B
Transformer layers: 28
Hidden-state index: 14
Hidden size: 1536
Extracted 10/73
Extracted 20/73
Extracted 30/73
Extracted 40/73
Extracted 50/73
Extracted 60/73
Extracted 70/73
Extracted 73/73

Hidden-state extraction complete
Feature matrix: (73, 1536)

TASK: CONFIG_3x3
Families: 5
Held out: CONFIG_3x3::DE::DE_T01
Held out: CONFIG_3x3::EN::EN_T01
Held out: CONFIG_3x3::EN::EN_T02
Held out: CONFIG_3x3::ZH::ZH_T01
Held out: CONFIG_3x3::ZH::ZH_T02

TASK: CONTEXT_SWAP_2x2
Families: 7
Held out: CONTEXT_SWAP_2x2::DE::DE_DRUCK
Held out: CONTEXT_SWAP_2x2::DE::DE_KONTROLLE
Held out: CONTEXT_SWAP_2x2::EN::EN_DOUBT
Held out: CONTEXT_SWAP_2x2::EN::EN_PROBLEM
Held out: CONTEXT_SWAP_2x2::ZH::ZH_CHANGE
Held out: CONTEXT_SWAP_2x2::ZH::ZH_DISCUSSION
Held out: CONTEXT_SWAP_2x2::ZH::ZH_INTEREST

Cross-validated predictions complete

====

In [1]:
from pathlib import Path
from collections import defaultdict
import csv, json, hashlib, random, statistics, math

ROOT = Path('/content') if Path('/content').exists() else Path('/mnt/data')
OUT = ROOT / 'mwe_v22_repro_audit'
OUT.mkdir(exist_ok=True)
REQUIRED = ['a2_scored_candidates.csv', 'b_scored_candidates.csv', 'b2_scored_candidates.csv', 'b2_folds.csv']

def locate(name):
    exact=ROOT/name
    if exact.exists(): return exact
    possibilities=[p for p in ROOT.glob(name.replace('.csv','*.csv')) if p.is_file()]
    return max(possibilities,key=lambda p:p.stat().st_mtime) if possibilities else None

missing=[x for x in REQUIRED if locate(x) is None]
if missing:
    try:
        from google.colab import files
    except ImportError as e:
        raise FileNotFoundError('Missing local inputs: '+', '.join(missing)) from e
    print('Please upload these missing files:', missing)
    files.upload()

PATHS={name: locate(name) for name in REQUIRED}
assert all(PATHS.values()),f'Missing input(s): {[k for k,v in PATHS.items() if v is None]}'

def read_csv(path):
    with open(path,encoding='utf-8-sig',newline='') as f:
        return list(csv.DictReader(f))

def write_csv(path,rows):
    assert rows, 'No rows to export: '+str(path)
    with open(path,'w',encoding='utf-8-sig',newline='') as f:
        w=csv.DictWriter(f,fieldnames=list(rows[0].keys()))
        w.writeheader(); w.writerows(rows)

a2=read_csv(PATHS['a2_scored_candidates.csv'])
b1=read_csv(PATHS['b_scored_candidates.csv'])
b2=read_csv(PATHS['b2_scored_candidates.csv'])
folds=read_csv(PATHS['b2_folds.csv'])
print('Input row counts:',len(a2),len(b1),len(b2),'folds',len(folds))
print('Input checksums:')
for name,path in PATHS.items():
    print(name, hashlib.sha256(path.read_bytes()).hexdigest()[:16])


Please upload these missing files: ['a2_scored_candidates.csv', 'b_scored_candidates.csv', 'b2_scored_candidates.csv', 'b2_folds.csv']


Saving a2_scored_candidates.csv to a2_scored_candidates.csv
Saving b_scored_candidates.csv to b_scored_candidates.csv
Saving b2_folds.csv to b2_folds.csv
Saving b2_scored_candidates.csv to b2_scored_candidates.csv
Input row counts: 73 73 73 folds 12
Input checksums:
a2_scored_candidates.csv ebc4e4f80ddead92
b_scored_candidates.csv a4ad8fc745c42ced
b2_scored_candidates.csv dfab0151d12039b3
b2_folds.csv 7c2eb46bd8c7429b


In [2]:
# Integrity checks: the same 73 candidates, 29 contexts, 12 construction families
for dataset in (a2,b1,b2):
    assert len(dataset)==73
    ids=[r['item_id'] for r in dataset]
    assert len(set(ids))==73
    assert all(r['item_validated']=='YES' for r in dataset)
by_id_a={r['item_id']:r for r in a2}
by_id_b={r['item_id']:r for r in b1}
by_id_h={r['item_id']:r for r in b2}
assert set(by_id_a)==set(by_id_b)==set(by_id_h)
key_fields=['task_type','language','group_id','context_id','candidate_id','context','completion','preferred','score_prior','score_lexical_overlap']
for item_id in by_id_a:
    for f in key_fields:
        assert by_id_a[item_id][f]==by_id_b[item_id][f]==by_id_h[item_id][f],(item_id,f)

contexts=defaultdict(list)
for r in b2:
    key=tuple(r[f] for f in ('task_type','language','group_id','context_id'))
    contexts[key].append(r)
families={(k[0],k[1],k[2]) for k in contexts}
assert len(contexts)==29 and len(families)==12
assert sum(len(v) for v in contexts.values())==73
assert len(folds)==12 and all(r['heldout_not_in_train']=='YES' for r in folds)
actual_folds={r['heldout_family']:r for r in folds}
assert len(actual_folds)==12
for task,lang,family in families:
    fold_key='::'.join([task,lang,family])
    assert fold_key in actual_folds
    assert int(actual_folds[fold_key]['test_rows'])==sum(len(v) for k,v in contexts.items() if k[:3]==(task,lang,family))
    assert int(actual_folds[fold_key]['train_rows'])== (45 if task=='CONFIG_3x3' else 28)-int(actual_folds[fold_key]['test_rows'])
print('PASS: 73 aligned items, 29 contexts, 12 LOFO families.')
# These fold-log checks cannot independently prove the original classifier had no unseen training leakage.


PASS: 73 aligned items, 29 contexts, 12 LOFO families.


In [3]:
METHODS = {
    'A2_PRIOR':'score_prior',
    'A2_TFIDF':'score_lexical_overlap',
    'B1_COND':'lm_cond_mean',
    'B1_LIFT':'lm_context_lift',
    'B2_LOFO':'probe_probability'
}

context_audit=[]
for key,rr in sorted(contexts.items()):
    task,lang,family,ctx=key
    gold=[r for r in rr if r['preferred']=='YES']
    assert len(gold)==1
    row={
        'task_type':task,'language':lang,'group_id':family,'context_id':ctx,
        'gold_candidate':gold[0]['candidate_id'],
        'context':gold[0]['context'],
    }
    for method,col in METHODS.items():
        candidates=[(r,float(r[col])) for r in rr]
        best=max(s for r,s in candidates)
        winners=[r for r,s in candidates if abs(s-best)<=1e-9]
        gold_score=float(gold[0][col])
        foil=max(float(r[col]) for r in rr if r['preferred']=='NO')
        ordered=sorted((s for _,s in candidates),reverse=True)
        row[method+'_winner']=';'.join(r['candidate_id'] for r in winners)
        row[method+'_top1_credit']=sum(r['preferred']=='YES' for r in winners)/len(winners)
        row[method+'_gold_minus_best_foil']=gold_score-foil  # corrected B1 error margin
        row[method+'_top2_gap']=ordered[0]-ordered[1]
        row[method+'_correct']=int(len(winners)==1 and winners[0]['preferred']=='YES')
    row['B2_near_tie_under_0_01']=int(row['B2_LOFO_top2_gap']<0.01)
    context_audit.append(row)

assert len(context_audit)==29
write_csv(OUT/'01_context_audit.csv',context_audit)
print('Corrected B1 margins and all model winners saved.')
print('B2 top-two gaps < 0.01:',sum(r['B2_near_tie_under_0_01'] for r in context_audit),'/29')


Corrected B1 margins and all model winners saved.
B2 top-two gaps < 0.01: 16 /29


In [4]:
family_audit=[]
family_rows=defaultdict(list)
for r in context_audit:
    family_rows[(r['task_type'],r['language'],r['group_id'])].append(r)
for key,rr in sorted(family_rows.items()):
    row={'task_type':key[0],'language':key[1],'group_id':key[2],'n_contexts':len(rr)}
    for m in METHODS:
        row[m+'_mean_top1']=statistics.mean(r[m+'_top1_credit'] for r in rr)
        row[m+'_all_contexts_correct']=int(all(r[m+'_correct']==1 for r in rr))
    row['B2_minus_B1_LIFT']=row['B2_LOFO_mean_top1']-row['B1_LIFT_mean_top1']
    family_audit.append(row)
write_csv(OUT/'02_family_audit.csv',family_audit)

summary=[]
for task,lang in sorted({(r['task_type'],r['language']) for r in context_audit}):
    rr=[r for r in context_audit if (r['task_type'],r['language'])==(task,lang)]
    for m in METHODS:
        summary.append({'task_type':task,'language':lang,'method':m,
                        'contexts':len(rr),'families':len({r['group_id'] for r in rr}),
                        'top1_credit_sum':sum(r[m+'_top1_credit'] for r in rr),
                        'context_top1_accuracy':statistics.mean(r[m+'_top1_credit'] for r in rr)})
for m in METHODS:
    rr=context_audit
    summary.append({'task_type':'ALL','language':'ALL','method':m,
                    'contexts':len(rr),'families':len(family_audit),
                    'top1_credit_sum':sum(r[m+'_top1_credit'] for r in rr),
                    'context_top1_accuracy':statistics.mean(r[m+'_top1_credit'] for r in rr)})
write_csv(OUT/'03_summary.csv',summary)
for r in summary:
    if r['task_type']=='ALL':print(r['method'],f"{r['top1_credit_sum']:g}/{r['contexts']}",f"{r['context_top1_accuracy']:.1%}")


A2_PRIOR 12/29 41.4%
A2_TFIDF 13/29 44.8%
B1_COND 18/29 62.1%
B1_LIFT 21/29 72.4%
B2_LOFO 25/29 86.2%


In [5]:
# Family-level paired bootstrap; family is the resampling unit, not the context.
# Exploratory percentile interval (descriptive, not definitive model comparison).
comparisons=[('B2_LOFO','B1_LIFT'),('B1_LIFT','A2_TFIDF'),('B1_COND','A2_TFIDF')]
ci_rows=[]
for left,right in comparisons:
    diffs=[r[left+'_mean_top1']-r[right+'_mean_top1'] for r in family_audit]
    seed=42
    rng=random.Random(seed)
    boot=sorted(statistics.mean(rng.choices(diffs,k=len(diffs))) for _ in range(20000))
    def pct(q):
        z=q*(len(boot)-1); i=int(z); w=z-i
        return boot[i]*(1-w)+boot[min(i+1,len(boot)-1)]*w
    ci_rows.append({'comparison':left+' minus '+right,'n_families':len(diffs),
                    'family_mean_difference':statistics.mean(diffs),
                    'bootstrap_95pct_low':pct(.025),'bootstrap_95pct_high':pct(.975),
                    'resamples':len(boot),'seed':seed})
write_csv(OUT/'04_family_bootstrap.csv',ci_rows)
for r in ci_rows:print(r['comparison'], 'mean',round(r['family_mean_difference'],4),
                       'CI', (round(r['bootstrap_95pct_low'],4),round(r['bootstrap_95pct_high'],4)))


B2_LOFO minus B1_LIFT mean 0.1528 CI (-0.0139, 0.3611)
B1_LIFT minus A2_TFIDF mean 0.2778 CI (0.0833, 0.4444)
B1_COND minus A2_TFIDF mean 0.1806 CI (-0.0139, 0.3889)


In [6]:
# Manual linguistic review: human-approved interpretive claims, NOT code-generated labels.
# Raw BCC corpus text is deliberately excluded from the public artifact pending license checks.
bcc_cases=[
    {'bcc_id':'B003','anchor_config':'BECOME','overt_outer_cause':'YES',
     'diagnostic':'Overt 使 + embedded 产生; candidate CAUSE-BECOME event-fragment linkage; he is matrix object and embedded subject',
     'interpretation_boundary':'Does not prove all silent vP layers; do not relabel embedded anchor as CAUSE',
     'source_status':'Manual review of BCC v1.2/v1.3 pilot; not a model prediction'},
    {'bcc_id':'B007','anchor_config':'CAUSE','overt_outer_cause':'NO',
     'diagnostic':'Lexical 引起; CAUSER is performance success, TOPIC is learning opera; disjoint',
     'interpretation_boundary':'No separately overt BECOME predicate; retain whole eventive causer NP',
     'source_status':'Manual review of BCC v1.2/v1.3 pilot; not a model prediction'},
    {'bcc_id':'B036','anchor_config':'CAUSE','overt_outer_cause':'NO',
     'diagnostic':'Lexical 引起; CAUSER and contextual TOPIC co-refer; TOPIC recovered from discourse',
     'interpretation_boundary':'Coreference is not proof of Si-style shared argument/chain linker',
     'source_status':'Manual review of BCC v1.2/v1.3 pilot; not a model prediction'},
]
write_csv(OUT/'05_bcc_manual_interpretation.csv',bcc_cases)
metadata={
    'audit_version':'v2.2',
    'statistical_unit':'12 construction families for paired bootstrap',
    'bootstrap_seed':42,'bootstrap_resamples':20000,
    'tie_tolerance':1e-9,'b2_near_tie_gap_threshold':0.01,
    'input_sha256':{k:hashlib.sha256(p.read_bytes()).hexdigest() for k,p in PATHS.items()},
    'provenance':'A2/B1/B2 outputs supplied by author; BCC interpretations manually reviewed, not model-inferred',
    'limitations':['No model inference or B2 feature extraction performed in this notebook',
                   'B2 supervised LOFO is not directly comparable to zero-shot B1',
                   'Explicit event/role cues may support semantic matching without syntactic recovery',
                   'Publication or redistribution of BCC raw sentences requires license review'],
}
(OUT/'06_metadata.json').write_text(json.dumps(metadata,ensure_ascii=False,indent=2),encoding='utf-8')
print('Audit complete. Files:',*[p.name for p in sorted(OUT.iterdir())],sep='\n • ')


Audit complete. Files:
 • 01_context_audit.csv
 • 02_family_audit.csv
 • 03_summary.csv
 • 04_family_bootstrap.csv
 • 05_bcc_manual_interpretation.csv
 • 06_metadata.json


In [7]:
# Optional: download computed artifacts individually (no ZIP).
try:
    from google.colab import files
    for p in sorted(OUT.iterdir()):
        files.download(str(p))
except ImportError:
    print('Local run: outputs saved to',OUT)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

... (2 outputs left out)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>